# Streaming Protein Pretraining

Main protein pretraining notebook. Training data is streamed from MinIO/S3 using `train_part_*.txt` shards. A local `tokenizer_map.json` is required before training starts. This notebook does not build a tokenizer from `train.txt`.

## 1. Imports

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import torch


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    MDCDecoderModel,
    build_mdc_config_from_progen_config,
    build_progen_config,
    compute_mdc_causal_lm_loss,
    count_trainable_parameters,
    create_streaming_protein_lm_dataloader,
    evaluate_mdc_causal_lm_batch_loss,
    generate_protein_text,
    prepare_mdc_training_runtime,
    save_protein_pretrain_checkpoint,
    set_mdc_data_loader_epoch,
)
from libs.core.pretrain.protein_lm.core import ProteinTokenizerArtifact
from libs.core.pretrain.training_config import (
    build_protein_training_data_config,
    create_protein_training_optimizer,
    describe_protein_training_optimizers,
    load_protein_training_config,
)
from libs.data.training.streaming import list_minio_text_parts
from libs.data.training.tokenizer import SequenceTokenizer

PROJECT_ROOT

## 2. Load Pretraining Config

In [ ]:
TRAINING_CONFIG = load_protein_training_config(PROJECT_ROOT)
PATHS_CONFIG = TRAINING_CONFIG["paths"]
DATA_CONFIG = TRAINING_CONFIG["data"]
MODEL_CONFIG = TRAINING_CONFIG["model"]
TRAINING_SETTINGS = TRAINING_CONFIG["training"]
OPTIMIZER_CONFIG = TRAINING_CONFIG["optimizer"]
RUNTIME_CONFIG = TRAINING_CONFIG["runtime"]
MINIO_CONFIG = TRAINING_CONFIG["minio"]
MINIO_DATA_CONFIG = build_protein_training_data_config(PROJECT_ROOT, TRAINING_CONFIG)

TOKENIZER_MAP_PATH = PATHS_CONFIG["tokenizer_map_path"]
CHECKPOINT_DIR = PATHS_CONFIG["checkpoint_dir"]
TRAIN_PART_CACHE_DIR = PATHS_CONFIG["train_part_cache_dir"]

MINIO_TRAIN_PARTS_PREFIX_URI = MINIO_CONFIG["train_parts_prefix_uri"]
MINIO_TRAIN_PART_URIS = MINIO_CONFIG["train_part_uris"]
if not MINIO_TRAIN_PARTS_PREFIX_URI and not MINIO_TRAIN_PART_URIS:
    raise ValueError("Set minio.train_parts_prefix_uri or minio.train_part_uris in train.yaml.")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PART_CACHE_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_LENGTH = MODEL_CONFIG["context_length"]
STRIDE = MODEL_CONFIG["stride"]
PROGEN_MODEL_SIZE = MODEL_CONFIG["progen_model_size"]
PROGEN_CONFIG_OVERRIDES = MODEL_CONFIG["progen_config_overrides"]

BATCH_SIZE = DATA_CONFIG["batch_size"]
NUM_WORKERS = DATA_CONFIG["num_workers"]
PIN_MEMORY = DATA_CONFIG["pin_memory"]
KEEP_DOWNLOADED_TRAIN_PARTS = DATA_CONFIG["keep_downloaded_train_parts"]

NUM_EPOCHS = TRAINING_SETTINGS["num_epochs"]
GRAD_CLIP_NORM = TRAINING_SETTINGS["grad_clip_norm"]
EVAL_FREQ = TRAINING_SETTINGS["eval_freq"]
EVAL_BATCHES = TRAINING_SETTINGS["eval_batches"]

REQUESTED_DEVICE = RUNTIME_CONFIG["device"]
MULTI_GPU_MODE = RUNTIME_CONFIG["multi_gpu_mode"]
DDP_FIND_UNUSED_PARAMETERS = RUNTIME_CONFIG["ddp_find_unused_parameters"]
DATA_PARALLEL_DEVICE_IDS = RUNTIME_CONFIG["data_parallel_device_ids"]

RANK = int(os.environ.get("RANK", "0"))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
IS_DISTRIBUTED = WORLD_SIZE > 1
IS_MAIN_PROCESS = RANK == 0

config_summary = {
    "tokenizer_map_path": str(TOKENIZER_MAP_PATH),
    "tokenizer_map_exists": TOKENIZER_MAP_PATH.exists(),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "train_part_cache_dir": str(TRAIN_PART_CACHE_DIR),
    "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
    "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "keep_downloaded_train_parts": KEEP_DOWNLOADED_TRAIN_PARTS,
    "device": REQUESTED_DEVICE,
    "world_size": WORLD_SIZE,
}
config_summary

## 3. Verify S3 Training Parts

This lists object metadata only. It does not download shard contents.

In [ ]:
s3_parts = list_minio_text_parts(
    prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
    part_uris=MINIO_TRAIN_PART_URIS or None,
    config=MINIO_DATA_CONFIG,
    suffixes=(".txt",),
)
if not s3_parts:
    raise FileNotFoundError("No S3 train_part_*.txt objects were found for streaming pretraining.")

part_summary = [
    {
        "index": index,
        "uri": part.uri,
        "size_bytes": part.size,
        "size_gib": None if part.size is None else round(part.size / 1024**3, 3),
    }
    for index, part in enumerate(s3_parts, start=1)
]
print(f"S3 train parts: {len(s3_parts)}")
print(f"Total GiB: {sum(part.size or 0 for part in s3_parts) / 1024**3:.3f}")
part_summary

## 4. Load Tokenizer

This step only loads `tokenizer_map.json`. If the file is missing, it raises an error and stops before model training.

In [ ]:
if MODEL_CONFIG["rebuild_tokenizer"]:
    raise ValueError("Streaming pretraining uses a provided tokenizer_map.json. Set model.rebuild_tokenizer=false.")
if not TOKENIZER_MAP_PATH.exists():
    raise FileNotFoundError(f"Provide tokenizer_map.json before pretraining: {TOKENIZER_MAP_PATH}")

tokenizer = SequenceTokenizer.load_map(TOKENIZER_MAP_PATH)
if tokenizer.sequence_type != "protein":
    raise ValueError("tokenizer_map.json must contain a protein tokenizer.")

tokenizer_artifact = ProteinTokenizerArtifact(
    tokenizer=tokenizer,
    tokenizer_map_path=TOKENIZER_MAP_PATH,
    rebuilt=False,
)

{
    "tokenizer_map_path": str(tokenizer_artifact.tokenizer_map_path),
    "vocab_size": tokenizer_artifact.vocab_size,
    "sequence_type": tokenizer_artifact.tokenizer.sequence_type,
}

## 5. Create Streaming Dataloaders

In [ ]:
loader_kwargs = {
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}

distributed_loader_kwargs = {
    "distributed": IS_DISTRIBUTED,
    "rank": RANK,
    "world_size": WORLD_SIZE,
}

train_loader = create_streaming_protein_lm_dataloader(
    tokenizer_artifact.tokenizer,
    prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
    part_uris=MINIO_TRAIN_PART_URIS or None,
    config=MINIO_DATA_CONFIG,
    cache_dir=TRAIN_PART_CACHE_DIR,
    keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
    shuffle_parts=True,
    shuffle_examples=True,
    seed=RANK,
    **loader_kwargs,
    **distributed_loader_kwargs,
)

train_eval_loader = (
    create_streaming_protein_lm_dataloader(
        tokenizer_artifact.tokenizer,
        prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
        part_uris=MINIO_TRAIN_PART_URIS or None,
        config=MINIO_DATA_CONFIG,
        cache_dir=TRAIN_PART_CACHE_DIR,
        keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
        shuffle_parts=False,
        shuffle_examples=False,
        seed=0,
        distributed=False,
        **loader_kwargs,
    )
    if IS_MAIN_PROCESS
    else None
)

print("Streaming pretraining dataloaders ready")
print(f"cache_dir={TRAIN_PART_CACHE_DIR}")

## 6. Build Model And Optimizer

In [ ]:
requested_device = REQUESTED_DEVICE
if requested_device == "auto":
    requested_device = "cuda" if torch.cuda.is_available() else "cpu"

runtime_dtype = torch.bfloat16 if requested_device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

progen_config = build_progen_config(
    PROGEN_MODEL_SIZE,
    vocab_size=tokenizer_artifact.vocab_size,
    context_length=CONTEXT_LENGTH,
    dtype=runtime_dtype,
    overrides=PROGEN_CONFIG_OVERRIDES,
)

model_config = build_mdc_config_from_progen_config(
    progen_config,
    dtype=runtime_dtype,
    attention_pattern="as_config",
)

runtime = prepare_mdc_training_runtime(
    MDCDecoderModel(model_config),
    device=requested_device,
    multi_gpu=MULTI_GPU_MODE,
    find_unused_parameters=DDP_FIND_UNUSED_PARAMETERS,
    data_parallel_device_ids=DATA_PARALLEL_DEVICE_IDS,
)
model = runtime.model
device = runtime.device

optimizer = create_protein_training_optimizer(model, OPTIMIZER_CONFIG, device=device)
optimizer_names = describe_protein_training_optimizers(optimizer)

runtime_summary = {
    "device": str(device),
    "dtype": str(runtime_dtype),
    "distributed": runtime.distributed,
    "data_parallel": runtime.data_parallel,
    "rank": runtime.rank,
    "world_size": runtime.world_size,
    "optimizer_types": optimizer_names,
    "trainable_parameters": count_trainable_parameters(model),
}
runtime_summary

## 7. Training Helpers

In [ ]:
checkpoint_last_path = CHECKPOINT_DIR / "checkpoint_last.pt"
checkpoint_best_path = CHECKPOINT_DIR / "checkpoint_best.pt"
metrics_path = CHECKPOINT_DIR / "streaming_training_metrics.jsonl"

global_step = 0
tokens_seen = 0
train_losses = []
val_losses = []
best_eval_loss = math.inf
optimizer_list = list(optimizer) if isinstance(optimizer, (list, tuple)) else [optimizer]


def move_batch(batch, device):
    return type(batch)(
        input_ids=batch.input_ids.to(device),
        attention_mask=batch.attention_mask.to(device),
        labels=batch.labels.to(device),
    )


def append_metrics(payload: dict[str, object]) -> None:
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")


def distributed_barrier() -> None:
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        torch.distributed.barrier()


def count_step_tokens(batch) -> int:
    token_count = torch.tensor(
        int(batch.attention_mask.sum().item()),
        device=device,
        dtype=torch.long,
    )
    if runtime.distributed:
        torch.distributed.all_reduce(token_count, op=torch.distributed.ReduceOp.SUM)
    return int(token_count.item())


def save_checkpoint(path: Path, epoch: int, eval_loss: float):
    return save_protein_pretrain_checkpoint(
        path,
        model=model,
        optimizer=optimizer,
        model_config=model_config,
        tokenizer=tokenizer_artifact.tokenizer,
        tokenizer_map_path=tokenizer_artifact.tokenizer_map_path,
        epoch=epoch,
        global_step=global_step,
        tokens_seen=tokens_seen,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=None if math.isinf(best_eval_loss) else best_eval_loss,
        training_args={
            "batch_size": BATCH_SIZE,
            "context_length": CONTEXT_LENGTH,
            "stride": STRIDE,
            "learning_rate": OPTIMIZER_CONFIG["learning_rate"],
            "muon_learning_rate": OPTIMIZER_CONFIG["muon_learning_rate"],
            "weight_decay": OPTIMIZER_CONFIG["weight_decay"],
            "optimizer_type": OPTIMIZER_CONFIG["type"],
            "optimizer_types": optimizer_names,
            "progen_model_size": PROGEN_MODEL_SIZE,
            "progen_config_overrides": PROGEN_CONFIG_OVERRIDES,
            "multi_gpu_mode": MULTI_GPU_MODE,
            "num_workers": NUM_WORKERS,
            "pin_memory": PIN_MEMORY,
            "train_config_path": str(TRAINING_CONFIG["config_path"]),
        },
        extra={
            "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
            "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
            "progen_config": dict(progen_config),
            "last_eval_loss": eval_loss,
        },
    )

print("Training helpers ready")

## 8. Streaming Pretrain

In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    set_mdc_data_loader_epoch(train_loader, epoch - 1)
    model.train()

    for batch in train_loader:
        batch = move_batch(batch, device)
        for opt in optimizer_list:
            opt.zero_grad(set_to_none=True)

        logits = model(batch.input_ids, batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        loss.backward()

        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        for opt in optimizer_list:
            opt.step()

        global_step += 1
        tokens_seen += count_step_tokens(batch)

        if runtime.is_main_process and EVAL_FREQ > 0 and global_step % EVAL_FREQ == 0:
            eval_loss = evaluate_mdc_causal_lm_batch_loss(
                model,
                train_eval_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            train_losses.append(eval_loss)
            val_losses.append(float("nan"))
            append_metrics({
                "epoch": epoch,
                "global_step": global_step,
                "tokens_seen": tokens_seen,
                "train_loss": eval_loss,
                "val_loss": float("nan"),
            })
            if eval_loss < best_eval_loss:
                best_eval_loss = eval_loss
                save_checkpoint(checkpoint_best_path, epoch, eval_loss)
            save_checkpoint(checkpoint_last_path, epoch, eval_loss)
            print(f"epoch={epoch} step={global_step} train={eval_loss:.4f} tokens_seen={tokens_seen:,}")

    distributed_barrier()

    if runtime.is_main_process:
        eval_loss = evaluate_mdc_causal_lm_batch_loss(
            model,
            train_eval_loader,
            device=device,
            max_batches=EVAL_BATCHES,
        )
        train_losses.append(eval_loss)
        val_losses.append(float("nan"))
        append_metrics({
            "epoch": epoch,
            "global_step": global_step,
            "tokens_seen": tokens_seen,
            "train_loss": eval_loss,
            "val_loss": float("nan"),
        })
        if eval_loss < best_eval_loss:
            best_eval_loss = eval_loss
            save_checkpoint(checkpoint_best_path, epoch, eval_loss)
        save_checkpoint(checkpoint_last_path, epoch, eval_loss)
        print(f"epoch={epoch} completed train={eval_loss:.4f} tokens_seen={tokens_seen:,}")

    distributed_barrier()

print(f"global_step={global_step} tokens_seen={tokens_seen:,}")
print(f"last checkpoint={checkpoint_last_path}")

## 9. Generate Sample

In [ ]:
sample = (
    generate_protein_text(
        model,
        tokenizer_artifact.tokenizer,
        prompt="<|protein|>",
        device=device,
        max_new_tokens=80,
        context_length=CONTEXT_LENGTH,
    )
    if runtime.is_main_process
    else f"Sample generation is emitted on rank 0 only. This rank is {runtime.rank}."
)
sample